# 🧪 Thí nghiệm 1: "Object đã xuất hiện trước đó bao nhiêu bước?"
### Đo lường xác suất xuất hiện sớm của object hallucinated vs. object thật trên LLaVA-1.5-7B

---

## 📋 Mục lục quy trình (Gate-by-Gate Roadmap)
1. **Cấu hình môi trường & Kiểm tra GPU**
2. **Tải Codebase từ GitHub & Cài đặt thư viện**
3. **Tải dữ liệu COCO 2014 (Annotations & Val Images kèm thanh tiến trình tqdm)**
4. **Cổng G0: Kiểm thử tự động (Unit Tests T1–T3)**
5. **Cổng G1: Smoke Test (3 ảnh) & Sanity GPU (T4–T6)**
6. **Cổng G2: Pilot Run (100 ảnh dev)**
7. **Cổng G3: Dev Run (500 ảnh) & Đánh giá CHAIR sanity**
8. **Cổng G4: Phân tích đầy đủ Dev (Câu hỏi A & B, Biểu đồ & Báo cáo nháp)**
9. **Cổng G5: Chạy tập Xác nhận Confirm (2000 ảnh) & Báo cáo cuối**
10. **Tải về kết quả (Download Zip Artifacts)**

> 💡 **Lưu ý:** Toàn bộ các bước xử lý nặng (sinh caption, gán nhãn, forward teacher-forcing, tính PMC) đều tích hợp sẵn thanh tiến trình **tqdm** hiển thị thời gian, tốc độ (img/s) và % hoàn thành.

## 1. Cấu hình môi trường & Kiểm tra GPU
Kiểm tra GPU đang hoạt động (Kaggle hỗ trợ GPU T4 x 2 hoặc P100 16GB).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Clone Codebase từ GitHub & Cài đặt Thư viện
Thay URL git repo của bạn vào bên dưới.

In [ ]:
import os

# TODO: Thay thế đường dẫn repo GitHub của bạn tại đây
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
REPO_DIR = "/kaggle/working/Experiment"

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q bitsandbytes  # Hỗ trợ 8-bit/4-bit nếu cần tiết kiệm VRAM

## 3. Chuẩn bị Dữ liệu COCO 2014 (Kèm thanh tiến trình tqdm)
Tự động tải annotations chính thức (`instances_val2014.json`, `captions_val2014.json`) và ảnh `val2014/` với thanh hiển thị tốc độ & phần trăm.

In [ ]:
import os
import urllib.request
from tqdm import tqdm

%cd {REPO_DIR}
os.makedirs("data", exist_ok=True)

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url, out_path):
    print(f"Downloading {os.path.basename(out_path)}...")
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(out_path), dynamic_ncols=True) as t:
        urllib.request.urlretrieve(url, filename=out_path, reporthook=t.update_to)

# 1. Tải COCO Annotations (~20MB)
if not os.path.exists("data/instances_val2014.json") or not os.path.exists("data/captions_val2014.json"):
    download_file("http://images.cocodataset.org/annotations/annotations_trainval2014.zip", "data/annotations.zip")
    !unzip -q -o data/annotations.zip annotations/instances_val2014.json annotations/captions_val2014.json -d data/
    !mv data/annotations/instances_val2014.json data/instances_val2014.json
    !mv data/annotations/captions_val2014.json data/captions_val2014.json
    !rm -rf data/annotations data/annotations.zip
    print("Annotations extracted successfully to data/")
else:
    print("Annotations already exist in data/")

# 2. Ảnh val2014 (6.2 GB):
# Nếu đã add dataset COCO vào Kaggle: `!ln -s /kaggle/input/coco-2014/val2014 data/val2014`
if not os.path.exists("data/val2014"):
    download_file("http://images.cocodataset.org/zips/val2014.zip", "data/val2014.zip")
    print("Extracting val2014.zip (vui lòng chờ trong giây lát)...")
    !unzip -q data/val2014.zip -d data/
    !rm -f data/val2014.zip
    print("val2014 images extracted to data/val2014/")
else:
    print("data/val2014/ already exists.")

## 4. Cổng G0: Kiểm thử tự động (Unit Tests T1–T3)
**Điều kiện qua:** Toàn bộ test T1 (token alignment, pieces_to_text, mention extraction), T2 (select_objects, caliper, seed determinism) và T3 (thống kê Bootstrap, Holm, Wilcoxon, AUROC) phải đạt 100%.

In [ ]:
%cd {REPO_DIR}
!python3 -m unittest discover -s tests -p "test_common.py" -v

## 5. Cổng G1: Smoke Test (3 ảnh) & GPU Sanity (T4–T6)
**Mục tiêu:** Chạy thử trên 3 ảnh đầu tiên để xác thực toàn bộ pipeline sinh caption, gán nhãn, và kiểm tra độ căn chỉnh vị trí token (`argmax(pred[i]) == gen_ids[i]` $\ge 98\%$).

In [ ]:
%cd {REPO_DIR}

# 1. Sinh caption cho 3 ảnh smoke (có thanh tiến trình tqdm)
!python3 src/01_generate.py --split smoke --n_images 3 --output_file data/smoke_captions.jsonl

# 2. Gán nhãn CHAIR cho 3 ảnh smoke
!python3 src/02_label_chair.py --captions_file data/smoke_captions.jsonl --output_file data/smoke_labels.jsonl

# In thử nội dung smoke
import json
with open("data/smoke_labels.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        print(f"Image ID {item['image_id']}: {len(item['mentions'])} mentions")
        for m in item['mentions']:
            print(f"   - {m['word']} (canon: {m['canon']}, tok: {m['tok_idx']}, halluc: {m['halluc']})")

# 3. Chạy GPU sanity check T4
!python3 -m unittest tests/test_gpu.py -v

## 6. Cổng G2: Pilot Run (100 ảnh dev)
**Mục tiêu:** Chạy qua 100 ảnh dev để kiểm tra toàn bộ pipeline từ Step 1 đến Step 5, đảm bảo sinh ra đầy đủ file và `funnel.json` hợp lý.

In [ ]:
%cd {REPO_DIR}

# Sinh caption 100 ảnh dev
!python3 src/01_generate.py --split dev --offset 0 --n_images 100 --output_file data/captions.jsonl

# Gán nhãn CHAIR
!python3 src/02_label_chair.py --captions_file data/captions.jsonl --output_file data/labels.jsonl

# Chạy lag curve (A)
!python3 src/03_lag_curve.py --labels_file data/labels.jsonl --output_dir results/exp1_pilot

# Chạy PMC (B)
!python3 src/04_pmc.py --labels_file data/labels.jsonl --captions_file data/captions.jsonl --output_dir results/exp1_pilot

# Trực quan hóa và báo cáo nháp pilot
!python3 src/05_visualize_report.py --captions_file data/captions.jsonl --labels_file data/labels.jsonl --output_dir results/exp1_pilot

# Xem funnel của Pilot
with open("results/exp1_pilot/funnel.json") as f:
    print(json.dumps(json.load(f), indent=2))

## 7. Cổng G3: Dev Run (500 ảnh) & CHAIR Sanity
Chạy tập dev 500 ảnh (`[0, 500)`). Kiểm tra `CHAIR_I` (kỳ vọng $\approx 6\%$, `CHAIR_S` $\approx 19.6\%$ đối chiếu thô).

In [ ]:
%cd {REPO_DIR}

# Sinh caption cho đủ 500 ảnh dev (tự động bỏ qua các ảnh đã sinh ở G2)
!python3 src/01_generate.py --split dev --offset 0 --n_images 500 --output_file data/captions.jsonl

# Gán nhãn CHAIR trên 500 ảnh dev
!python3 src/02_label_chair.py --captions_file data/captions.jsonl --output_file data/labels.jsonl

## 8. Cổng G4: Kết quả Dev Đầy Đủ (A & B) & Báo Cáo Nháp
Chạy forward teacher-forcing, phân tích ghép cặp đối chứng, phân tích PMC và xuất báo cáo nháp cùng các biểu đồ.

In [ ]:
%cd {REPO_DIR}

# 1. Chạy Câu hỏi A: Lag curve forward teacher forcing
!python3 src/03_lag_curve.py --labels_file data/labels.jsonl --output_dir results/exp1

# 2. Chạy Câu hỏi B: PMC analysis
!python3 src/04_pmc.py --labels_file data/labels.jsonl --captions_file data/captions.jsonl --output_dir results/exp1

# 3. Tạo biểu đồ và REPORT.md
!python3 src/05_visualize_report.py --captions_file data/captions.jsonl --labels_file data/labels.jsonl --output_dir results/exp1

In [ ]:
# Hiển thị các biểu đồ kết quả
from IPython.display import Image, display
import os

figs = [
    "results/exp1/figures/s_vs_m.png",
    "results/exp1/figures/delta_vs_m.png",
    "results/exp1/figures/auroc_vs_m.png",
    "results/exp1/figures/pmc_by_group.png",
    "results/exp1/figures/pmc_by_nprec.png",
]

for f in figs:
    if os.path.exists(f):
        print(f"=== {f} ===")
        display(Image(filename=f))

# In nội dung báo cáo REPORT.md
if os.path.exists("results/exp1/REPORT.md"):
    with open("results/exp1/REPORT.md") as r_file:
        print(r_file.read()[:2000])

## 9. Cổng G5: Chạy tập Xác nhận Confirm (`[500, 2500)`)
> ⚠️ **Chỉ chạy Cổng G5 sau khi G0–G4 đã hoàn tất và kết quả dev ổn định.**
Chạy xác nhận trên 2.000 ảnh độc lập với mọi tham số đã cố định.

In [ ]:
%cd {REPO_DIR}

# Sinh caption cho tập confirm [500, 2500)
!python3 src/01_generate.py --split confirm --offset 500 --n_images 2000 --output_file data/captions_confirm.jsonl

# Gán nhãn CHAIR cho confirm
!python3 src/02_label_chair.py --captions_file data/captions_confirm.jsonl --output_file data/labels_confirm.jsonl

# Chạy phân tích A
!python3 src/03_lag_curve.py --labels_file data/labels_confirm.jsonl --output_dir results/exp1_confirm

# Chạy phân tích B (PMC)
!python3 src/04_pmc.py --labels_file data/labels_confirm.jsonl --captions_file data/captions_confirm.jsonl --output_dir results/exp1_confirm

# Tạo biểu đồ & REPORT.md chung cuộc cho tập confirm
!python3 src/05_visualize_report.py --captions_file data/captions_confirm.jsonl --labels_file data/labels_confirm.jsonl --output_dir results/exp1_confirm

## 10. Tải về Kết quả (Download Zip Artifacts)
Nén toàn bộ kết quả, biểu đồ và báo cáo thành file zip để tải về lưu trữ.

In [ ]:
%cd {REPO_DIR}
!zip -r /kaggle/working/experiment1_results.zip results/
print("Tập tin kết quả đã được nén tại: /kaggle/working/experiment1_results.zip")
print("Bạn có thể tải trực tiếp từ tab 'Output' bên phải giao diện Kaggle!")